# 05 — Demo (hi→mr)

Gradio-free demo: translates a few hardcoded Hindi sentences with the fine-tuned
adapter and prints results. If `BODHAN_API_KEY` is set in the environment, the
Bodhan hosted API is also queried as an extra comparison (guarded; skipped otherwise).

In [ ]:
import sys
sys.path.insert(0, "../src")

CONFIG = "../configs/base.yaml"
ADAPTER = ""
FAMILY = "bodhan"  # "bodhan" | "indictrans2"

SENTENCES = [
    "शिक्षा हर बच्चे का अधिकार है।",
    "महाराष्ट्र की राजधानी मुंबई है।",
    "कृपया मुझे पुस्तकालय का रास्ता बताइए।",
    "किसान खेत में धान की फसल काट रहे हैं।",
]

from mr_mt.config import load_config
from mr_mt.evaluate import load_model_for_family
from mr_mt.inference import translate

cfg = load_config(CONFIG)
model, tokenizer = load_model_for_family(cfg, ADAPTER, FAMILY)
for s in SENTENCES:
    print("HI:", s)
    print("MR:", translate(s, model, tokenizer, cfg, FAMILY))
    print()

In [ ]:
"""Optional Bodhan hosted-API comparison (skipped unless BODHAN_API_KEY is set)."""
import json
import os
import urllib.request

api_key = os.environ.get("BODHAN_API_KEY", "")
api_url = os.environ.get("BODHAN_API_URL", "")  # set alongside the key when available
if not api_key or not api_url:
    print("BODHAN_API_KEY/BODHAN_API_URL not set — skipping hosted comparison.")
else:
    for s in SENTENCES:
        payload = json.dumps({"input": s, "source_lang": "hin_Deva",
                              "target_lang": "mar_Deva"}).encode("utf-8")
        req = urllib.request.Request(
            api_url, data=payload,
            headers={"Content-Type": "application/json", "Authorization": f"Bearer {api_key}"},
        )
        try:
            with urllib.request.urlopen(req, timeout=60) as resp:
                print("HI:", s)
                print("BODHAN-API:", resp.read().decode("utf-8"))
        except Exception as exc:
            print(f"hosted API call failed for {s!r}: {exc}")